In [ ]:
# 🎯 Example 1: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”

# ================================
# AGENT 1: PLANNER
# ================================
def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]


# ================================
# AGENT 2: FLIGHT AGENT
# ================================
def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]


# ================================
# AGENT 3: WEATHER AGENT
# ================================
def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result


# RUN
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response)

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


In [ ]:
# 🎯 Example 2: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: PLANNER (INTAKE AGENT)
# ================================
def planner_agent(user_query):
    print("\n[Planner Agent] Creating Plan...")
    return ["tests", "treatment", "decision"]


# ================================
# AGENT 2: DIAGNOSTIC AGENT
# ================================
def diagnostic_agent():
    print("\n[Diagnostic Agent] Diagnosing Patient Condition...")
    return [
        {"test": "Blood Test", "result": "Normal"},
        {"test": "X-Ray", "result": "No Findings"}
    ]


# ================================
# AGENT 3: TREATMENT AGENT
# ================================
def treatment_agent():
    print("\n[Treatment Agent] Suggesting Treatment Options...")
    return ["Medication A", "Therapy B"]


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(test, treatment_options):
    print("\n[Decision Agent] Making Decision...")

    # check first test result
    first_test = test[0]

    if first_test["result"] == "Normal":
        return "No need for therapy, patient is normal"

    return f"Take proper treatment: {treatment_options}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def hospital_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    test = None
    treatment_options = None

    for step in plan:

        if step == "tests":
            test = diagnostic_agent()

        elif step == "treatment":
            treatment_options = treatment_agent()

        elif step == "decision":
            result = decision_agent(test, treatment_options)

    return result


# ================================
# RUN
# ================================
response = hospital_multi_agent("I am suffering from fever")
print("\nFinal Answer:", response)

User Query: I am suffering from fever

[Planner Agent] Creating Plan...

[Diagnostic Agent] Diagnosing Patient Condition...

[Treatment Agent] Suggesting Treatment Options...

[Decision Agent] Making Decision...

Final Answer: No need for therapy, patient is normal


In [39]:
import requests
from google.colab import userdata

# ================================
# 🔑 HUGGINGFACE API
# ================================

API_TOKEN = userdata.get("HUGGINFACE_KEY")

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}


# ================================
# LLM QUERY FUNCTION
# ================================
def query_llm(prompt):

    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 150
    }

    response = requests.post(
        API_URL,
        headers=headers,
        json=payload
    )

    try:
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except:
        return response.text


# ================================
# TOOLS
# ================================

def blood_test():
    return {"test": "Blood Test", "result": "Normal"}


def xray_test():
    return {"test": "X-Ray", "result": "No Findings"}


def give_medicine():
    return "Medication A"


def give_therapy():
    return "Therapy B"


# ================================
# AGENT 1 — PLANNER
# ================================
def planner_agent(user_query):

    thought = query_llm(
        f"Create hospital plan for patient: {user_query}"
    )

    print("\n[Planner Agent Thought]")
    print(thought)

    return ["tests", "treatment", "decision"]


# ================================
# AGENT 2 — DIAGNOSTIC
# ================================
def diagnostic_agent(user_query):

    thought = query_llm(
        f"Suggest medical tests for: {user_query}"
    )

    print("\n[Diagnostic Agent Thought]")
    print(thought)

    return [
        blood_test(),
        xray_test()
    ]


# ================================
# AGENT 3 — TREATMENT
# ================================
def treatment_agent(user_query):

    thought = query_llm(
        f"Suggest treatment for: {user_query}"
    )

    print("\n[Treatment Agent Thought]")
    print(thought)

    return [
        give_medicine(),
        give_therapy()
    ]


# ================================
# AGENT 4 — DECISION
# ================================
def decision_agent(test_results, treatments):

    prompt = f"""
    Test Results: {test_results}
    Treatments: {treatments}

    Give final diagnosis and treatment.
    """

    print("\n[Decision Agent]")

    return query_llm(prompt)


# ================================
# MAIN SYSTEM
# ================================
def hospital_multi_agent(user_query):

    print("\nUser Query:", user_query)

    plan = planner_agent(user_query)

    tests = None
    treatments = None

    for step in plan:

        if step == "tests":
            tests = diagnostic_agent(user_query)

        elif step == "treatment":
            treatments = treatment_agent(user_query)

        elif step == "decision":
            final = decision_agent(
                tests,
                treatments
            )

    return final


# ================================
# TEST
# ================================

print(
    hospital_multi_agent(
        "I have fever and headache"
    )
)

print(
    hospital_multi_agent(
        "Chest pain and cough"
    )
)


User Query: I have fever and headache

[Planner Agent Thought]
**Hospital Admission Plan for Patient**

**Patient Information:**

1. Name: [Insert Patient's Name]
2. Age: [Insert Patient's Age]
3. Contact Information: [Insert Patient's Contact Information]

**Chief Complaints:**

1. Fever
2. Headache

**History of Present Illness:**

- The patient presents with fever and headache.
- The duration of symptoms is [Insert Duration of Symptoms].
- The patient's symptoms started [Insert Time of Onset].
- The patient's symptoms worsened [Insert Time of Worsening].
- Associated symptoms: [Insert Associated Symptoms].

**Past Medical History:**

1. List all past medical conditions:
    - [Insert Past Medical Conditions]
2. List

[Diagnostic Agent Thought]
If you have a fever and headache, it's essential to undergo medical tests to determine the underlying cause. Here are some suggested medical tests:

**Initial Tests:**

1. **Complete Blood Count (CBC)**: This test measures various components 

In [ ]:
# Example 2: Manager–Worker Multi-Agent System
# 🧠 Scenario

# “Now instead of fixed flow, we introduce a Manager Agent
# that assigns tasks dynamically to worker

# ================================
# WORKER AGENTS
# ================================
def flight_agent():
    print("[Flight Agent] Working...")
    return [{"airline": "IndiGo", "price": 4500},
            {"airline": "Air India", "price": 5200}]


def weather_agent():
    print("[Weather Agent] Working...")
    return {"condition": "Clear", "temp": 28}


# ================================
# MANAGER AGENT
# ================================
def manager_agent(user_query):
    print("\n[Manager Agent] Analyzing task...")

    tasks = []

    if "trip" in user_query.lower():
        tasks = ["flight", "weather"]

    return tasks


# ================================
# EXECUTION
# ================================
def run_system(user_query):
    print("User Query:", user_query)

    tasks = manager_agent(user_query)

    results = {}

    for task in tasks:
        if task == "flight":
            results["flights"] = flight_agent()

        elif task == "weather":
            results["weather"] = weather_agent()

    # Final decision
    cheapest = min(results["flights"], key=lambda x: x["price"])

    return f"Manager Decision: Book {cheapest['airline']} at ₹{cheapest['price']}"


# RUN
response = run_system("Plan my trip")
print("\nFinal Answer:", response)


User Query: Plan my trip

[Manager Agent] Analyzing task...
[Flight Agent] Working...
[Weather Agent] Working...

Final Answer: Manager Decision: Book IndiGo at ₹4500


In [ ]:
# =====================================
# Scenario: Corporate Market Research
# Multi-Agent System (No API)
# =====================================


# ===============================
# WORKER 1 — Market Research
# ===============================
def market_worker():
    print("\n[Market Worker] Researching market...")
    return "High demand in Tier-1 cities, moderate competition"


# ===============================
# WORKER 2 — Finance
# ===============================
def finance_worker():
    print("\n[Finance Worker] Checking budget...")
    return "Investment needed: $10M, ROI in 2 years"


# ===============================
# WORKER 3 — Operations
# ===============================
def operations_worker():
    print("\n[Operations Worker] Checking supply chain...")
    return "Factories ready, shipping cost high"


# ===============================
# WORKER 4 — Legal
# ===============================
def legal_worker():
    print("\n[Legal Worker] Checking regulations...")
    return "Trademark available, certification required"


# ===============================
# WORKER 5 — HR
# ===============================
def hr_worker():
    print("\n[HR Worker] Checking staffing...")
    return "Need 50 new employees"


# ===============================
# MANAGER AGENT
# ===============================
def manager_agent(goal):

    print("Goal:", goal)

    reports = {}

    # Manager decides tasks dynamically
    tasks = ["market", "finance", "operations", "legal", "hr"]

    for task in tasks:

        if task == "market":
            reports["market"] = market_worker()

        elif task == "finance":
            reports["finance"] = finance_worker()

        elif task == "operations":
            reports["operations"] = operations_worker()

        elif task == "legal":
            reports["legal"] = legal_worker()

        elif task == "hr":
            reports["hr"] = hr_worker()

    # Final decision
    print("\n[Manager] Final Decision...")

    if "high" in reports["market"].lower():
        return "Launch Product Y in Asia with planning"
    else:
        return "Not feasible to launch"


# ===============================
# RUN
# ===============================
result = manager_agent(
    "Evaluate feasibility of launching Product Y in Asia"
)

print("\nFinal Result:", result)

Goal: Evaluate feasibility of launching Product Y in Asia

[Market Worker] Researching market...

[Finance Worker] Checking budget...

[Operations Worker] Checking supply chain...

[Legal Worker] Checking regulations...

[HR Worker] Checking staffing...

[Manager] Final Decision...

Final Result: Launch Product Y in Asia with planning


In [ ]:
client = Groq(api_key="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")


In [ ]:
!pip install groq
from groq import Groq

from google.colab import userdata

API_KEY = userdata.get("API_KEY")

import nest_asyncio
nest_asyncio.apply()

import asyncio
from groq import Groq


# ================================
# GROQ CLIENT
# ================================
client = Groq(api_key=API_KEY)



# ================================
# LLM QUERY
# ================================
def query_llm(prompt):

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content


# ================================
# AGENT 1 – Planner (Manager Plan)
# ================================
def planner_agent(goal):

    print("\n[Planner Agent] Creating workflow...")

    prompt = f"""
    Goal: {goal}

    What departments should evaluate this?
    """

    thought = query_llm(prompt)

    print(thought)

    return ["market", "finance", "operations", "legal", "hr", "decision"]


# ================================
# AGENT 2 – Market Worker
# ================================
def market_worker(goal):

    print("\n[Market Worker] Researching market...")

    prompt = f"""
    Goal: {goal}

    Do market research.
    """

    return query_llm(prompt)


# ================================
# AGENT 3 – Finance Worker
# ================================
def finance_worker(goal):

    print("\n[Finance Worker] Checking budget...")

    prompt = f"""
    Goal: {goal}

    Check financial feasibility.
    """

    return query_llm(prompt)


# ================================
# AGENT 4 – Operations Worker
# ================================
def operations_worker(goal):

    print("\n[Operations Worker] Checking supply chain...")

    prompt = f"""
    Goal: {goal}

    Check operations readiness.
    """

    return query_llm(prompt)


# ================================
# AGENT 5 – Legal Worker
# ================================
def legal_worker(goal):

    print("\n[Legal Worker] Checking regulations...")

    prompt = f"""
    Goal: {goal}

    Check legal requirements.
    """

    return query_llm(prompt)


# ================================
# AGENT 6 – HR Worker
# ================================
def hr_worker(goal):

    print("\n[HR Worker] Checking staffing...")

    prompt = f"""
    Goal: {goal}

    Check HR requirements.
    """

    return query_llm(prompt)


# ================================
# AGENT 7 – Decision Agent
# ================================
def decision_agent(reports):

    print("\n[Decision Agent] Final decision...")

    prompt = f"""
    Market: {reports['market']}

    Finance: {reports['finance']}

    Operations: {reports['operations']}

    Legal: {reports['legal']}

    HR: {reports['hr']}

    Should company launch the product?
    """

    return query_llm(prompt)


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def corporate_multi_agent(goal):

    print("Goal:", goal)

    plan = planner_agent(goal)

    reports = {}

    for step in plan:

        if step == "market":
            reports["market"] = market_worker(goal)

        elif step == "finance":
            reports["finance"] = finance_worker(goal)

        elif step == "operations":
            reports["operations"] = operations_worker(goal)

        elif step == "legal":
            reports["legal"] = legal_worker(goal)

        elif step == "hr":
            reports["hr"] = hr_worker(goal)

        elif step == "decision":
            result = decision_agent(reports)

    return result


# ================================
# RUN SYSTEM
# ================================

response = corporate_multi_agent(
    "Evaluate feasibility of launching Product Y in Asia"
)

print("\nFinal Answer:\n", response)


Goal: Evaluate feasibility of launching Product Y in Asia

[Planner Agent] Creating workflow...
To evaluate the feasibility of launching Product Y in Asia, the following departments should be involved:

1. **Market Research Department**: They can conduct market analysis and gather data on demand, competition, market trends, and customer preferences in Asia.
2. **Product Development Department**: They can assess the product's suitability for the Asian market, considering factors such as local regulations, product features, and pricing strategies.
3. **Sales and Marketing Department**: They can evaluate the sales and marketing strategies, channel distribution, and customer acquisition plans for the Asian market.
4. **Regulatory and Compliance Department**: They can ensure that Product Y complies with local regulations, laws, and standards in Asia, such as product labeling, packaging, and safety requirements.
5. **Finance Department**: They can assess the funding requirements, revenue pro

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("API_KEY")

import nest_asyncio
nest_asyncio.apply()

import asyncio
from groq import Groq


# ================================
# GROQ CLIENT
# ================================
client = Groq(api_key=API_KEY)


# ================================
# LLM QUERY
# ================================
def query_llm(prompt):

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content


# ================================
# AGENT 1 – Crisis Coordinator (Broadcast)
# ================================
def broadcaster_agent():

    message = (
        "Data breach detected in customer database. "
        "Immediate response required."
    )

    print("\n[Coordinator] Broadcasting crisis message...")

    return message


# ================================
# AGENT 2 – IT Security
# ================================
def it_agent(message):

    print("\n[IT Security Agent]")

    prompt = f"""
    Crisis message: {message}

    What IT security actions needed?
    """

    return query_llm(prompt)


# ================================
# AGENT 3 – Communications
# ================================
def communication_agent(message):

    print("\n[Communication Agent]")

    prompt = f"""
    Crisis message: {message}

    What communication steps needed?
    """

    return query_llm(prompt)


# ================================
# AGENT 4 – Finance
# ================================
def finance_agent(message):

    print("\n[Finance Agent]")

    prompt = f"""
    Crisis message: {message}

    What financial actions needed?
    """

    return query_llm(prompt)


# ================================
# AGENT 5 – Legal
# ================================
def legal_agent(message):

    print("\n[Legal Agent]")

    prompt = f"""
    Crisis message: {message}

    What legal steps required?
    """

    return query_llm(prompt)


# ================================
# AGENT 6 – HR
# ================================
def hr_agent(message):

    print("\n[HR Agent]")

    prompt = f"""
    Crisis message: {message}

    What HR actions needed?
    """

    return query_llm(prompt)


# ================================
# AGENT 7 – Decision Agent
# ================================
def decision_agent(responses):

    print("\n[Decision Agent] Creating final plan...")

    prompt = f"""
    IT: {responses['it']}

    Communication: {responses['comm']}

    Finance: {responses['finance']}

    Legal: {responses['legal']}

    HR: {responses['hr']}

    Create final crisis response plan.
    """

    return query_llm(prompt)


# ================================
# MAIN MULTI AGENT SYSTEM
# ================================
def crisis_multi_agent():

    message = broadcaster_agent()

    responses = {}

    # Broadcast to all agents
    responses["it"] = it_agent(message)
    responses["comm"] = communication_agent(message)
    responses["finance"] = finance_agent(message)
    responses["legal"] = legal_agent(message)
    responses["hr"] = hr_agent(message)

    final = decision_agent(responses)

    return final


# ================================
# RUN SYSTEM
# ================================
result = crisis_multi_agent()

print("\nFinal Crisis Plan:\n", result)



[Coordinator] Broadcasting crisis message...

[IT Security Agent]

[Communication Agent]

[Finance Agent]

[Legal Agent]

[HR Agent]

[Decision Agent] Creating final plan...

Final Crisis Plan:
 **Crisis Response Plan: Data Breach in Customer Database**

**I. Introduction**

The purpose of this crisis response plan is to outline the procedures for responding to a data breach in the customer database. The plan is designed to minimize the impact of the breach, protect the company's reputation, and ensure compliance with relevant laws and regulations.

**II. Crisis Management Structure**

The crisis management team will be responsible for implementing and managing the response to the data breach. The team will consist of the following representatives:

* CEO/Executive Leadership
* CISO (Chief Information Security Officer)
* IT Team
* Compliance Officer
* Public Relations Specialist
* Law Enforcement Liaison (if necessary)

**III. Immediate Response (Within 1 Hour)**

1. **Notification of

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("API_KEY")

import nest_asyncio
nest_asyncio.apply()

import asyncio
from groq import Groq


# ================================
# GROQ CLIENT
# ================================
client = Groq(api_key=API_KEY)


# ================================
# LLM QUERY
# ================================
def query_llm(prompt):

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content


# ================================
# AGENTS (ASYNC)
# ================================

async def marketing_agent(msg):

    print("[Marketing Agent]")

    prompt = f"""
    Announcement: {msg}

    Create marketing strategy.
    """

    return query_llm(prompt)


async def finance_agent(msg):

    print("[Finance Agent]")

    prompt = f"""
    Announcement: {msg}

    Create finance plan.
    """

    return query_llm(prompt)


async def operations_agent(msg):

    print("[Operations Agent]")

    prompt = f"""
    Announcement: {msg}

    Create operations plan.
    """

    return query_llm(prompt)


async def legal_agent(msg):

    print("[Legal Agent]")

    prompt = f"""
    Announcement: {msg}

    Check legal requirements.
    """

    return query_llm(prompt)


async def hr_agent(msg):

    print("[HR Agent]")

    prompt = f"""
    Announcement: {msg}

    Create HR plan.
    """

    return query_llm(prompt)


# ================================
# DECISION AGENT
# ================================
def decision_agent(responses):

    print("\n[Decision Agent] Final plan...")

    prompt = f"""
    Marketing: {responses[0]}

    Finance: {responses[1]}

    Operations: {responses[2]}

    Legal: {responses[3]}

    HR: {responses[4]}

    Create final product launch plan.
    """

    return query_llm(prompt)


# ================================
# BROADCAST / COORDINATOR
# ================================
async def broadcast():

    message = (
        "Product X launch in Q3, "
        "target market North America, "
        "budget 5 million dollars"
    )

    print("\n[Coordinator] Broadcasting launch plan...\n")

    responses = await asyncio.gather(

        marketing_agent(message),
        finance_agent(message),
        operations_agent(message),
        legal_agent(message),
        hr_agent(message)

    )

    final = decision_agent(responses)

    print("\nFinal Launch Plan:\n")
    print(final)


# ================================
# RUN
# ================================
asyncio.run(broadcast())


[Coordinator] Broadcasting launch plan...

[Marketing Agent]
[Finance Agent]
[Operations Agent]
[Legal Agent]
[HR Agent]

[Decision Agent] Final plan...

Final Launch Plan:

**Final Product Launch Plan**

**Product X Launch Plan:**

**Executive Summary:**

Our comprehensive plan for the launch of Product X in the North American market will focus on creating a strong online presence, driving traffic and sales, and establishing Product X as a leader in the industry.

**Marketing Strategy:**

Our marketing strategy includes:

* Digital Marketing (40% of budget):
 + Website development and SEO optimization
 + Social media marketing (Facebook, LinkedIn, Twitter, and Instagram) with targeted ads and content creation
 + Email marketing campaigns (welcome series, nurture campaigns, and abandoned cart reminders)
 + Influencer marketing (partner with 10-15 industry influencers for product reviews and testimonials)
* Content Marketing (20% of budget):
 + Develop engaging content (blog posts, whi

In [ ]:
# Goal → Orchestrator → Search → Analyst → Writer → QA → Final Report

In [41]:
from google.colab import userdata

API_KEY = userdata.get("API_KEY")

import nest_asyncio
nest_asyncio.apply()

import asyncio
from groq import Groq


# ================================
# GROQ CLIENT
# ================================
client = Groq(api_key=API_KEY)

# ===============================
# SHARED MEMORY
# ===============================
class SharedMemory:

    def __init__(self):
        self.storage = {}

    def update(self, agent, data):
        self.storage[agent] = data

    def get(self, agent):
        return self.storage.get(agent)

    def all(self):
        return self.storage


memory = SharedMemory()


# ===============================
# LLM CALL FUNCTION
# ===============================
def query_llm(prompt):

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


# ===============================
# SEARCH AGENT
# ===============================
class SearchAgent:

    name = "search"

    def run(self, goal, memory):

        print("\n[Search Agent] Gathering EV industry information...")

        prompt = f"""
        Goal: {goal}

        Collect key EV market facts and trends in India for 2025.
        """

        result = query_llm(prompt)

        return result


# ===============================
# ANALYST AGENT
# ===============================
class AnalystAgent:

    name = "analyst"

    def run(self, goal, memory):

        print("\n[Analyst Agent] Interpreting data...")

        search_data = memory.get("search")

        prompt = f"""
        Analyze the following EV market data:

        {search_data}

        Provide key insights and trends.
        """

        return query_llm(prompt)


# ===============================
# WRITER AGENT
# ===============================
class WriterAgent:

    name = "writer"

    def run(self, goal, memory):

        print("\n[Writer Agent] Drafting report...")

        analysis = memory.get("analyst")

        prompt = f"""
        Using the analysis below, write a structured report.

        {analysis}

        Topic: EV industry trends in India for 2025.
        """

        return query_llm(prompt)


# ===============================
# QA AGENT
# ===============================
class QAAgent:

    name = "qa"

    def run(self, goal, memory):

        print("\n[QA Agent] Checking report quality...")

        draft = memory.get("writer")

        prompt = f"""
        Review this report for accuracy, clarity, and tone.

        {draft}

        Suggest improvements or corrections.
        """

        return query_llm(prompt)


# ===============================
# ORCHESTRATOR
# ===============================
def orchestrator(goal):

    print("Goal:", goal)

    search = SearchAgent()
    analyst = AnalystAgent()
    writer = WriterAgent()
    qa = QAAgent()

    agents = [search, analyst, writer, qa]

    for agent in agents:

        result = agent.run(goal, memory)

        memory.update(agent.name, result)

    final_report = memory.get("writer")

    print("\n==============================")
    print("FINAL REPORT")
    print("==============================\n")

    print(final_report)

    print("\n==============================")
    print("QA FEEDBACK")
    print("==============================\n")

    print(memory.get("qa"))


# ===============================
# RUN PIPELINE
# ===============================
goal = "Write a comprehensive market report on EV industry trends in India for 2025."

orchestrator(goal)

Goal: Write a comprehensive market report on EV industry trends in India for 2025.

[Search Agent] Gathering EV industry information...

[Analyst Agent] Interpreting data...

[Writer Agent] Drafting report...

[QA Agent] Checking report quality...

FINAL REPORT

**Structured Report: EV Industry Trends in India for 2025**

**Executive Summary:**
The Indian EV market is expected to witness rapid growth in 2025, driven by government initiatives, increasing adoption of electric vehicles among consumers, and investments in the EV ecosystem. The market is expected to grow at a CAGR of 50% and reach $150 billion by 2025. This report provides an analysis of the key trends, drivers, and challenges in the Indian EV market, along with recommendations for manufacturers and investors.

**Introduction:**
The Indian EV market is on the cusp of rapid growth, driven by government initiatives, increasing adoption of electric vehicles among consumers, and investments in the EV ecosystem. This report aims